In [2]:
import yfinance as yf

# Lista completa dei ticker USA
tickers_usa = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "JPM", "JNJ", "XOM", "PG", "KO",
    "V", "HD", "CAT", "NEE", "TSLA"
]

# Ciclo di estrazione per ogni singolo titolo
for symbol in tickers_usa:
    print(f"--- Dati anagrafici per {symbol} ---")

    # Inizializzazione del ticker e chiamata API per l'anagrafica
    ticker = yf.Ticker(symbol)
    info = ticker.info

    # Mapping per la tabella core.instruments
    instrument_data = {
        "instrument_name": info.get("longName"),
        "short_name": info.get("shortName"),
        "instrument_type": info.get("quoteType"),
        "ticker": info.get("symbol"),
        "trading_currency_code": info.get("currency")
    }

    # Mapping per la tabella core.instrument_classification
    classification_data = {
        "sector": info.get("sector"),
        "industry": info.get("industry")
    }

    # Stampa dei risultati formattati
    print("Dati core.instruments:", instrument_data)
    print("Dati core.instrument_classification:", classification_data)
    print("-" * 40 + "\n")

--- Dati anagrafici per AAPL ---
Dati core.instruments: {'instrument_name': 'Apple Inc.', 'short_name': 'Apple Inc.', 'instrument_type': 'EQUITY', 'ticker': 'AAPL', 'trading_currency_code': 'USD'}
Dati core.instrument_classification: {'sector': 'Technology', 'industry': 'Consumer Electronics'}
----------------------------------------

--- Dati anagrafici per MSFT ---
Dati core.instruments: {'instrument_name': 'Microsoft Corporation', 'short_name': 'Microsoft Corporation', 'instrument_type': 'EQUITY', 'ticker': 'MSFT', 'trading_currency_code': 'USD'}
Dati core.instrument_classification: {'sector': 'Technology', 'industry': 'Software - Infrastructure'}
----------------------------------------

--- Dati anagrafici per NVDA ---
Dati core.instruments: {'instrument_name': 'NVIDIA Corporation', 'short_name': 'NVIDIA Corporation', 'instrument_type': 'EQUITY', 'ticker': 'NVDA', 'trading_currency_code': 'USD'}
Dati core.instrument_classification: {'sector': 'Technology', 'industry': 'Semiconduct

Scaricare e caricamento dei prezzi

solo per i seguenti titoli:

        - "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "JPM", "JNJ", "XOM",
        "PG", "KO", "V", "HD", "CAT", "NEE", "TSLA"

        - Petrolifero ed Energetico (USA, IT, ES, DE)
        "CVX", "COP", "SLB", "ENI.MI", "ENEL.MI",
        "REP.MC", "IBE.MC", "RWE.DE", "EOAN.DE", "ENR.DE"

        - Difesa / Armamenti (USA, IT)
        "LMT", "RTX", "NOC", "GD", "LDO.MI", "FCT.MI",

        - Alimentare e Supermercati USA
        "WMT", "COST", "KR", "PEP", "MDLZ", "GIS",

        - Crypto e MicroStrategy
        "BTC-USD", "ETH-USD", "MSTR",

        - Infrastrutture
        "AMT", "EQIX", "UNP", "PWR", "TRN.MI", "SRG.MI"

In [6]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import yfinance as yf
from datetime import datetime

# 1. Caricamento variabili d'ambiente
load_dotenv()

def get_connection():
    """Crea e restituisce la connessione al database PostgreSQL."""
    return psycopg2.connect(
        host=os.getenv("PGHOST"),
        database=os.getenv("PGDATABASE"),
        user=os.getenv("PGUSER"),
        password=os.getenv("PGPASSWORD"),
        sslmode=os.getenv("PGSSLMODE")
    )

# 2. Funzione per il Bulk Insert
def inserisci_prezzi(df):
    """
    Inserisce un DataFrame di Pandas nel database usando execute_values
    per garantire le massime prestazioni nel Bulk Insert.
    """
    if df.empty:
        print("Nessun dato da inserire.")
        return

    # Sostituisce i valori NaN (dati mancanti in Pandas) con None (NULL in PostgreSQL)
    df = df.where(pd.notnull(df), None)

    conn = None
    cur = None

    try:
        conn = get_connection()
        cur = conn.cursor()

        # NOME DELLA TABELLA CORRETTO CON LO SCHEMA
        table_name = "raw.instrument_prices"
        columns = [
            "ticker", "isin", "price_date", "open_price", "high_price",
            "low_price", "close_price", "adjusted_close", "volume",
            "currency", "source_name", "source_file", "batch_id", "loaded_at"
        ]

        # Converte il DataFrame in una lista di tuple per psycopg2
        valori = [tuple(x) for x in df[columns].to_numpy()]

        # Query dinamica per il bulk insert
        query_insert = f"""
            INSERT INTO {table_name} ({', '.join(columns)})
            VALUES %s
        """

        # Esegue un singolo commit massivo
        execute_values(cur, query_insert, valori)
        conn.commit()

        print(f"✓ Sweet! {len(df)} righe di dati aggiornati/inseriti in '{table_name}'.")

    except Exception as e:
        if conn:
            conn.rollback()
        print("Errore durante l'inserimento:", e)

    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()

# 3. Funzione di scaricamento dati
def scarica_storico_yfinance():
    """Scarica lo storico dei prezzi dal 2010 al 2025 da Yahoo Finance."""
    tickers_list = [
        "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "JPM", "JNJ", "XOM",
        "PG", "KO", "V", "HD", "CAT", "NEE", "TSLA",
        # Petrolifero ed Energetico (USA, IT, ES, DE)
        "CVX", "COP", "SLB", "ENI.MI", "ENEL.MI",
        "REP.MC", "IBE.MC", "RWE.DE", "EOAN.DE", "ENR.DE",
        # Difesa / Armamenti (USA, IT)
        "LMT", "RTX", "NOC", "GD", "LDO.MI", "FCT.MI",
        # Alimentare e Supermercati USA
        "WMT", "COST", "KR", "PEP", "MDLZ", "GIS",
        # Crypto e MicroStrategy
        "BTC-USD", "ETH-USD", "MSTR",
        # Infrastrutture
        "AMT", "EQIX", "UNP", "PWR", "TRN.MI", "SRG.MI"
    ]

    start_date = "2010-01-01"
    end_date = "2026-01-01" # Esclusivo, copre fino al 31-12-2025

    # Genera un numero intero basato sulla data e ora, es: 20260925004520:
    batch_id_generato = int(datetime.now().strftime('%Y%m%d%H%M%S'))

    # ---> CORREZIONE AGGIUNTA QUI <---
    timestamp_caricamento = datetime.now()

    print("Scaricamento dati da Yahoo Finance in corso...")
    tutti_i_dati = []

    for ticker in tickers_list:
        try:
            df_ticker = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)

            if not df_ticker.empty:
                df_ticker = df_ticker.reset_index()

                # Appiattisce le colonne se Yahoo Finance restituisce un MultiIndex
                if isinstance(df_ticker.columns, pd.MultiIndex):
                    df_ticker.columns = df_ticker.columns.get_level_values(0)

                # Aggiunta colonne statiche
                df_ticker['ticker'] = ticker
                df_ticker['isin'] = ""
                df_ticker['currency'] = "USD"
                df_ticker['source_name'] = "Yahoo Finance"
                df_ticker['source_file'] = "API_Call"
                df_ticker['batch_id'] = batch_id_generato
                df_ticker['loaded_at'] = timestamp_caricamento

                # Rinomina per il database
                df_ticker = df_ticker.rename(columns={
                    "Date": "price_date", "Open": "open_price", "High": "high_price",
                    "Low": "low_price", "Close": "close_price", "Adj Close": "adjusted_close",
                    "Volume": "volume"
                })

                # Formattazione data in stringa YYYY-MM-DD
                df_ticker['price_date'] = df_ticker['price_date'].dt.strftime('%Y-%m-%d')

                tutti_i_dati.append(df_ticker)
                print(f"[{ticker}] - OK")
            else:
                print(f"[{ticker}] - Nessun dato trovato.")

        except Exception as e:
            print(f"[{ticker}] - Errore: {e}")

    if tutti_i_dati:
        df_completo = pd.concat(tutti_i_dati, ignore_index=True)
        return df_completo

    return pd.DataFrame()

# --- ESECUZIONE DEL PROGRAMMA ---
if __name__ == "__main__":
    df_prezzi = scarica_storico_yfinance()

    if not df_prezzi.empty:
        inserisci_prezzi(df_prezzi)
    else:
        print("Operazione annullata: nessun dato scaricato.")

Scaricamento dati da Yahoo Finance in corso...
[AAPL] - OK
[MSFT] - OK
[NVDA] - OK
[AMZN] - OK
[GOOGL] - OK
[JPM] - OK
[JNJ] - OK
[XOM] - OK
[PG] - OK
[KO] - OK
[V] - OK
[HD] - OK
[CAT] - OK
[NEE] - OK
[TSLA] - OK
[CVX] - OK
[COP] - OK
[SLB] - OK
[ENI.MI] - OK
[ENEL.MI] - OK
[REP.MC] - OK
[IBE.MC] - OK
[RWE.DE] - OK
[EOAN.DE] - OK
[ENR.DE] - OK
[LMT] - OK
[RTX] - OK
[NOC] - OK
[GD] - OK
[LDO.MI] - OK
[FCT.MI] - OK
[WMT] - OK
[COST] - OK
[KR] - OK
[PEP] - OK
[MDLZ] - OK
[GIS] - OK
[BTC-USD] - OK
[ETH-USD] - OK
[MSTR] - OK
[AMT] - OK
[EQIX] - OK
[UNP] - OK
[PWR] - OK
[TRN.MI] - OK
[SRG.MI] - OK
✓ Sweet! 180662 righe di dati aggiornati/inseriti in 'raw.instrument_prices'.
